<a href="https://colab.research.google.com/github/dhrupsarkar965/Implementing-ML-Algorithms/blob/main/Student_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [262]:
import pandas as pd


In [263]:
df=pd.read_csv("/content/StudentPerformanceFactors.csv")

In [264]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


#Basic EDA

In [265]:
df.shape

(6607, 20)

In [266]:
df.isnull().sum()

,0
Hours_Studied,0
Attendance,0
Parental_Involvement,0
Access_to_Resources,0
Extracurricular_Activities,0
Sleep_Hours,0
Previous_Scores,0
Motivation_Level,0
Internet_Access,0
Tutoring_Sessions,0


In [267]:
df.duplicated().sum()

np.int64(0)

In [268]:
df.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [269]:
df.corr(numeric_only=True)

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
Hours_Studied,1.000000,-0.009908,0.010977,0.024846,-0.014282,0.004624,0.445455
Attendance,-0.009908,1.000000,-0.015918,-0.020186,0.014324,-0.022435,0.581072
Sleep_Hours,0.010977,-0.015918,1.000000,-0.021750,-0.012216,-0.000378,-0.017022
Previous_Scores,0.024846,-0.020186,-0.021750,1.000000,-0.013122,-0.011274,0.175079
Tutoring_Sessions,-0.014282,0.014324,-0.012216,-0.013122,1.000000,0.017733,0.156525
Physical_Activity,0.004624,-0.022435,-0.000378,-0.011274,0.017733,1.000000,0.027824
Exam_Score,0.445455,0.581072,-0.017022,0.175079,0.156525,0.027824,1.000000


#Now training a model

In [270]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDRegressor
from sklearn.impute import SimpleImputer

In [271]:
df.head(2)

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61


In [272]:
x=df.drop(columns=['Exam_Score'])
y=df['Exam_Score']

In [273]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [274]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

In [275]:
#separate numerical and categoical column
x_numeric=x_train[['Hours_Studied','Attendance','Sleep_Hours','Previous_Scores','Tutoring_Sessions','Physical_Activity']]
x_categorical=x_train.drop(columns=['Hours_Studied','Attendance','Sleep_Hours','Previous_Scores','Tutoring_Sessions','Physical_Activity'])

In [276]:
numerical_pipeline=Pipeline(steps=[
    ('standard_scaler',StandardScaler())
])

In [277]:
x_categorical.columns

Index(['Parental_Involvement', 'Access_to_Resources',
       'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access',
       'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence',
       'Learning_Disabilities', 'Parental_Education_Level',
       'Distance_from_Home', 'Gender'],
      dtype='object')

In [278]:
print('parentalinvo:',x_categorical['Parental_Involvement'].unique())
print('Access_to_Resources:',x_categorical['Access_to_Resources'].unique())
print('Extracurricular_Activities:',x_categorical['Extracurricular_Activities'].unique())
print("Motivation:",x_categorical['Motivation_Level'].unique())
print('Internet_Access:',x_categorical['Internet_Access'].unique())
print('Family_Income:',x_categorical['Family_Income'].unique())
print('Teacher_Quality:',x_categorical['Teacher_Quality'].unique())
print('School_Typ:',x_categorical['School_Type'].unique())
print('Peer_Influence:',x_categorical['Peer_Influence'].unique())
print('Learning_Disabilities:',x_categorical['Learning_Disabilities'].unique())
print('Parental_Education_Level:',x_categorical['Parental_Education_Level'].unique())
print('Distance_from_Home:',x_categorical['Distance_from_Home'].unique())
print('Gender:',x_categorical['Gender'].unique())



parentalinvo: ['Low' 'High' 'Medium']
Access_to_Resources: ['High' 'Medium' 'Low']
Extracurricular_Activities: ['Yes' 'No']
Motivation: ['High' 'Medium' 'Low']
Internet_Access: ['Yes' 'No']
Family_Income: ['Low' 'Medium' 'High']
Teacher_Quality: ['Medium' 'High' nan 'Low']
School_Typ: ['Public' 'Private']
Peer_Influence: ['Negative' 'Neutral' 'Positive']
Learning_Disabilities: ['No' 'Yes']
Parental_Education_Level: ['College' 'High School' 'Postgraduate' nan]
Distance_from_Home: ['Moderate' 'Far' 'Near' nan]
Gender: ['Female' 'Male']


In [279]:
categorical_pipeline=Pipeline(steps=[
     ('imputer',SimpleImputer(strategy='most_frequent')),
     ('ordinal_encoder',OrdinalEncoder(categories=[['Low','Medium','High'],['Low','Medium','High'],['No','Yes'],['Low','Medium','High'],['No','Yes'],['Low','Medium','High'],['Low','Medium','High'],['Public','Private'],['Negative','Neutral','Positive'],['No','Yes'],['High School','College','Postgraduate'],['Far','Moderate','Near']]))

])

In [280]:
categorical_pipeline=Pipeline(steps=[
     ('imputer',SimpleImputer(strategy='most_frequent')),
     ('ordinal_encoder',OrdinalEncoder(categories=[['Low','Medium','High'],['Low','Medium','High'],['No','Yes'],['Low','Medium','High'],['No','Yes'],['Low','Medium','High'],['Low','Medium','High'],['Public','Private'],['Negative','Neutral','Positive'],['No','Yes'],['High School','College','Postgraduate'],['Far','Moderate','Near']]))
])

In [281]:
numerical_pipeline=Pipeline(steps=[
    ('standard_scaler',StandardScaler())
])

In [282]:
x_categorical.columns

Index(['Parental_Involvement', 'Access_to_Resources',
       'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access',
       'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence',
       'Learning_Disabilities', 'Parental_Education_Level',
       'Distance_from_Home', 'Gender'],
      dtype='object')

In [283]:
column_transformer=ColumnTransformer(transformers=[
    ('numerical_pipeline',numerical_pipeline,x_numeric.columns),
    ('categorical_pipeline',categorical_pipeline,['Parental_Involvement', 'Access_to_Resources',
       'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access',
       'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence',
       'Learning_Disabilities', 'Parental_Education_Level',
       'Distance_from_Home']),
    ('categorical_pipeline1',OneHotEncoder(handle_unknown='ignore'),['Gender'])
],remainder='passthrough')

In [284]:
reg=LinearRegression()

In [285]:
from sklearn.pipeline import make_pipeline
pipefinal=make_pipeline(column_transformer,reg)

In [286]:
pipefinal.fit(x_train,y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical_pipeline',
                                                  Pipeline(steps=[('standard_scaler',
                                                                   StandardScaler())]),
                                                  Index(['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores',
       'Tutoring_Sessions', 'Physical_Activity'],
      dtype='object')),
                                                 ('categorical_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   Sim...
                                                  ['Parental_Involvement',
                                                   'Access_to_Resources',
                                                   'Extracurricular_Activities',
                                                   'Motivation_Level',
                                                   'Internet_Access',
                                                   'Family_Income',
                                                   'Teacher_Quality',
                                                   'School_Type',
                                                   'Peer_Influence',
                                                   'Learning_Disabilities',
                                                   'Parental_Education_Level',
                                                   'Distance_from_Home']),
                                                 ('categorical_pipeline1',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender'])])),
                ('linearregression', LinearRegression())])

In [287]:
pipefinal.score(x_test,y_test)

0.7709282415480216

In [288]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [289]:
y_pred = pipefinal.predict(x_test)

In [290]:
mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

MAE: 0.44428945627208355
RMSE: 1.7994293282174323


In [294]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipefinal,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores)
print("Average R2:", scores.mean())

CV Scores: [0.79254374 0.81088044 0.74066602 0.70984135 0.56432622]
Average R2: 0.7236515539600636


In [292]:
#this code just to check if we reduce our no column does prediction improve or not

from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

# 1. Initialize PCA with the desired number of components
pca = PCA(n_components=19)

# 2. Build the final pipeline with Preprocessing -> PCA -> Linear Regression
pipe_with_pca = make_pipeline(
    column_transformer,
    pca,

)

# 3. Fit the pipeline with PCA on your training data
pca_xtrain=pipe_with_pca.fit_transform(x_train)
pca_xtest=pipe_with_pca.transform(x_test)

# 4. Evaluate the new R2 score on the test data
reg.fit(pca_xtrain,y_train)
pca_score = reg.score(pca_xtest, y_test)
pca_score

0.7709282415480216